# 02. Discounted REINFORCE on CartPole

A companion to the [post-training primer](https://www.norange.ai/post-training/2026/08/24/post-training-primer.html). Run all cells in order in a **Colab CPU runtime**; no GPU or account tokens are needed.

Unlike the bandit, this agent makes a sequence of decisions: push a cart left or right to balance a pole. CartPole gives +1 per step, including the step on which it fails. Its observation contains cart position and velocity and pole angle and angular velocity.

We train a small neural policy directly with REINFORCE, then plot learning and animate the policy before and after training. Training is stochastic; improvement is not a guarantee for every seed.

In [ ]:
%pip -q install "gymnasium[classic-control]==1.1.1" "torch>=2.2,<3" "matplotlib>=3.7,<4"

In [ ]:
import os
os.environ["SDL_VIDEODRIVER"] = "dummy"  # Headless rendering in Colab.
os.environ["SDL_AUDIODRIVER"] = "dummy"

import copy
import numpy as np
import torch
from torch import nn
from torch.distributions import Categorical
import gymnasium as gym
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

torch.set_num_threads(1)
SEED = 7
np.random.seed(SEED)
torch.manual_seed(SEED)
GAMMA = 0.99
MAX_STEPS = 500
BATCH_EPISODES = 16
UPDATES = 120
LEARNING_RATE = 0.01

def make_env(render_mode=None):
    return gym.make("CartPole-v1", max_episode_steps=MAX_STEPS,
                    render_mode=render_mode)

policy = nn.Sequential(nn.Linear(4, 32), nn.Tanh(), nn.Linear(32, 2))
initial_policy = copy.deepcopy(policy)
optimizer = torch.optim.Adam(policy.parameters(), lr=LEARNING_RATE)

## The objective and the exact episode loss

We define a **finite-horizon task**: an episode ends at failure or after 500 steps. The time limit is part of this task, so there is no reward beyond it and no value bootstrap at truncation.

$$J_\gamma(\theta)=\mathbb E\left[\sum_{t=0}^{T-1}\gamma^t r_t\right],\qquad
G_t^{(\gamma)}=\sum_{k=t}^{T-1}\gamma^{k-t}r_k.$$

Compute the local return backward, using $G_T=0$ and $G_t=r_t+\gamma G_{t+1}$. The policy loss for an episode is

$$L(\theta)=-\sum_{t=0}^{T-1}\gamma^t G_t^{(\gamma)}\log\pi_\theta(a_t\mid s_t).$$

The **outer $\gamma^t$ matters**: the local return discounts from time $t$, while the objective discounts from the beginning of the episode. Returns are treated as fixed targets. We sum over steps and average over episodes, without return normalization, a learned baseline, or entropy bonuses, to match the derivation directly. A batch of episodes reduces sampling noise.

With $\gamma=1$, this becomes undiscounted REINFORCE. CartPole illustrates the estimator; a task with competing short- and long-term goals would better illustrate how discounting changes preferred behavior.

In [ ]:
def discounted_returns(rewards, gamma):
    returns = np.empty(len(rewards), dtype=np.float32)
    running = 0.0
    for t in reversed(range(len(rewards))):
        running = float(rewards[t]) + gamma * running
        returns[t] = running
    return torch.from_numpy(returns)

# A hand-computable trajectory checks local and episode-start discounting.
example = discounted_returns([1, 2, 3], 0.5)
torch.testing.assert_close(example, torch.tensor([2.75, 3.5, 3.0]))
torch.testing.assert_close(
    (0.5 ** torch.arange(3)) * example,
    torch.tensor([2.75, 1.75, 0.75]),
)
torch.testing.assert_close(
    discounted_returns([1, 2, 3], 1.0), torch.tensor([6., 5., 3.])
)

In [ ]:
def collect_episode(env, model, seed=None):
    observation, _ = env.reset(seed=seed)
    log_probs, rewards = [], []
    while True:
        state = torch.as_tensor(observation, dtype=torch.float32)
        distribution = Categorical(logits=model(state))
        action = distribution.sample()
        log_probs.append(distribution.log_prob(action))
        observation, reward, terminated, truncated, _ = env.step(action.item())
        rewards.append(reward)
        if terminated or truncated:
            break
    return torch.stack(log_probs), rewards

@torch.no_grad()
def evaluate(model, seeds):
    # The same reset seeds and action RNG stream make comparisons reproducible.
    lengths = []
    env = make_env()
    try:
        with torch.random.fork_rng():
            torch.manual_seed(SEED + 10000)
            for seed in seeds:
                _, rewards = collect_episode(env, model, seed=int(seed))
                lengths.append(len(rewards))
    finally:
        env.close()
    return np.asarray(lengths)

EVAL_SEEDS = range(1000, 1020)
before_lengths = evaluate(initial_policy, EVAL_SEEDS)
print(f"Initial policy: {before_lengths.mean():.1f} steps on average")

## Train with complete episodes

Each update samples fresh trajectories from the current stochastic policy. The plots show batch averages, not a claim that every episode improves. The undiscounted episode length is useful for judging balance, while the discounted return is the objective we optimize.

In [ ]:
history = {"steps": [], "discounted_return": [], "loss": []}
env = make_env()
try:
    for update in range(UPDATES):
        episode_losses, lengths, objectives = [], [], []
        for episode in range(BATCH_EPISODES):
            seed = SEED if update == 0 and episode == 0 else None
            log_probs, rewards = collect_episode(env, policy, seed=seed)
            returns = discounted_returns(rewards, GAMMA)
            discounts = GAMMA ** torch.arange(len(rewards), dtype=torch.float32)
            episode_losses.append(-(discounts * returns * log_probs).sum())
            lengths.append(len(rewards))
            objectives.append(returns[0].item())
        loss = torch.stack(episode_losses).mean()
        optimizer.zero_grad()
        loss.backward()
        if not torch.isfinite(loss) or any(
            not torch.isfinite(p.grad).all() for p in policy.parameters()
        ):
            raise RuntimeError("Non-finite loss or gradient")
        optimizer.step()
        history["steps"].append(np.mean(lengths))
        history["discounted_return"].append(np.mean(objectives))
        history["loss"].append(loss.item())
        if (update + 1) % 10 == 0:
            print(f"Update {update + 1:3d}/{UPDATES}: "
                  f"length={np.mean(lengths):6.1f}, "
                  f"discounted return={np.mean(objectives):6.2f}")
finally:
    env.close()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, key, label in zip(axes, ["steps", "discounted_return"],
                          ["Episode length", "Discounted episode return"]):
    ax.plot(np.arange(1, UPDATES + 1), history[key], color="#7c3aed")
    ax.set(xlabel="Policy update", ylabel=label)
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

after_lengths = evaluate(policy, EVAL_SEEDS)
print(f"Before: {before_lengths.mean():.1f} ± {before_lengths.std():.1f} steps")
print(f"After:  {after_lengths.mean():.1f} ± {after_lengths.std():.1f} steps")
print("Mean ± standard deviation over 20 evaluation episodes; cap = 500 steps.")

## Watch before and after training

Both videos start from the same reset seed and sample actions from their policies. These are illustrative rollouts, not cherry-picked successes; use the evaluation averages above to assess learning. Frames are subsampled to keep the notebook small.

In [ ]:
@torch.no_grad()
def animate_policy(model, title, seed=2026):
    env = make_env(render_mode="rgb_array")
    frames = []
    try:
        observation, _ = env.reset(seed=seed)
        frames.append(env.render())
        with torch.random.fork_rng():
            torch.manual_seed(seed)
            for step in range(MAX_STEPS):
                distribution = Categorical(logits=model(torch.as_tensor(
                    observation, dtype=torch.float32)))
                observation, _, terminated, truncated, _ = env.step(
                    distribution.sample().item())
                if (step + 1) % 3 == 0 or terminated or truncated:
                    frames.append(env.render())
                if terminated or truncated:
                    break
    finally:
        env.close()
    fig, ax = plt.subplots(figsize=(6, 4))
    artist = ax.imshow(frames[0])
    ax.set_title(f"{title} — {step + 1} steps")
    ax.axis("off")
    def draw(i):
        artist.set_data(frames[i])
        return (artist,)
    animation = FuncAnimation(fig, draw, frames=len(frames), interval=60, blit=True)
    html = animation.to_jshtml(default_mode="once")
    plt.close(fig)
    return HTML(html)

plt.rcParams["animation.embed_limit"] = 60

In [ ]:
display(animate_policy(initial_policy, "Before training"))
display(animate_policy(policy, "After training"))

## Experiments and references

- Change `GAMMA` to `1.0` or `0.95`, then rerun from the setup cell to reset the policy and optimizer. Compare several seeds: a single run cannot establish which setting learns better.
- Inspect the discounted return and episode length together. Discounted returns saturate as episode length grows, particularly for smaller discount factors.
- Keep the outer discount in the loss when experimenting: dropping it would no longer implement the exact episodic gradient derived in the post.

Sources: [Gymnasium CartPole documentation](https://gymnasium.farama.org/environments/classic_control/cart_pole/); [Sutton and Barto, Reinforcement Learning: An Introduction](https://incompleteideas.net/book/the-book-2nd.html), Chapters 3 and 13.